# Baseline Models

This notebook trains simple baselines:
- majority class
- logistic regression
- random forest

The sequence input is flattened from `[30, 13]` to `[390]`.

In [4]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.dataset import StockSequenceDataset, print_dataset_summary

def rel_path(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return path.as_posix()

print("Setup complete.")

Setup complete.


## Config

In [5]:
CSV_PATH = PROJECT_ROOT / "data" / "processed" / "features.csv"

LOOKBACK = 30
SEED = 42

MAX_TRAIN_SAMPLES = 250_000
MAX_VAL_SAMPLES = 120_000

LABEL_NAMES = ["down", "flat", "up"]

RF_N_ESTIMATORS = 200
RF_MAX_DEPTH = 12
RF_MIN_SAMPLES_LEAF = 50
RF_N_JOBS = -1

print("CSV exists:", CSV_PATH.exists())

CSV exists: True


## Load and flatten data

In [6]:
train_dataset = StockSequenceDataset(str(CSV_PATH), split="train", lookback=LOOKBACK)
val_dataset = StockSequenceDataset(str(CSV_PATH), split="val", lookback=LOOKBACK)

print_dataset_summary(train_dataset, "Train Dataset")
print_dataset_summary(val_dataset, "Validation Dataset")

def sample_indices(n, max_samples=None, seed=42):
    if max_samples is None or n <= max_samples:
        return np.arange(n)
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=max_samples, replace=False)
    return np.sort(idx)

train_idx = sample_indices(len(train_dataset), MAX_TRAIN_SAMPLES, seed=SEED)
val_idx = sample_indices(len(val_dataset), MAX_VAL_SAMPLES, seed=SEED)

X_train = train_dataset.X[train_idx].reshape(len(train_idx), -1)
y_train = train_dataset.y[train_idx]

X_val = val_dataset.X[val_idx].reshape(len(val_idx), -1)
y_val = val_dataset.y[val_idx]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

[train] Number of feature columns: 13
[train] Feature columns: ['daily_return', 'log_return', 'open_close_return', 'high_low_range', 'volume_change', 'ma_5_ratio', 'ma_10_ratio', 'ma_20_ratio', 'volatility_5', 'volatility_10', 'volatility_20', 'momentum_5', 'momentum_10']
[val] Number of feature columns: 13
[val] Feature columns: ['daily_return', 'log_return', 'open_close_return', 'high_low_range', 'volume_change', 'ma_5_ratio', 'ma_10_ratio', 'ma_20_ratio', 'volatility_5', 'volatility_10', 'volatility_20', 'momentum_5', 'momentum_10']
Train Dataset
Number of samples: 1005533
X shape: (1005533, 30, 13)
y shape: (1005533,)
Target distribution:
  0: 312017 (31.03%)
  1: 283953 (28.24%)
  2: 409563 (40.73%)
Date range:
2010-03-17 00:00:00 to 2018-12-31 00:00:00
Number of symbols:
478
Number of feature columns:
13
Validation Dataset
Number of samples: 368455
X shape: (368455, 30, 13)
y shape: (368455,)
Target distribution:
  0: 116598 (31.65%)
  1: 87428 (23.73%)
  2: 164429 (44.63%)
Date 

## Train and evaluate

In [7]:
def evaluate_predictions(model_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    cm = confusion_matrix(y_true, y_pred)

    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)
    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Weighted F1:", weighted_f1)
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))
    print("Confusion matrix:")
    print(cm)

    return {
        "model": model_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "confusion_matrix": cm,
    }

results = []

models = [
    ("Majority Class", DummyClassifier(strategy="most_frequent")),
    ("Logistic Regression", Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            solver="lbfgs",
            max_iter=300,
            C=1.0,
            n_jobs=-1,
            random_state=SEED,
        )),
    ])),
    ("Random Forest", RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        n_jobs=RF_N_JOBS,
        random_state=SEED,
        class_weight=None,
    )),
]

for name, model in models:
    start = time.time()
    model.fit(X_train, y_train)
    train_seconds = time.time() - start
    pred = model.predict(X_val)
    result = evaluate_predictions(name, y_val, pred)
    result["train_seconds"] = train_seconds
    results.append(result)


Majority Class
Accuracy: 0.446325
Macro F1: 0.20572831140995282
Weighted F1: 0.27546506577014157
              precision    recall  f1-score   support

        down     0.0000    0.0000    0.0000     37912
        flat     0.0000    0.0000    0.0000     28529
          up     0.4463    1.0000    0.6172     53559

    accuracy                         0.4463    120000
   macro avg     0.1488    0.3333    0.2057    120000
weighted avg     0.1992    0.4463    0.2755    120000

Confusion matrix:
[[    0     0 37912]
 [    0     0 28529]
 [    0     0 53559]]


c:\Users\51454\anaconda3\envs\stock\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\51454\anaconda3\envs\stock\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\51454\anaconda3\envs\stock\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape


Logistic Regression
Accuracy: 0.43433333333333335
Macro F1: 0.3226398103035441
Weighted F1: 0.36789493224684583
              precision    recall  f1-score   support

        down     0.3617    0.1557    0.2177     37912
        flat     0.3714    0.1113    0.1713     28529
          up     0.4525    0.8036    0.5790     53559

    accuracy                         0.4343    120000
   macro avg     0.3952    0.3569    0.3226    120000
weighted avg     0.4045    0.4343    0.3679    120000

Confusion matrix:
[[ 5902  2232 29778]
 [ 3044  3176 22309]
 [ 7373  3144 43042]]

Random Forest
Accuracy: 0.44495833333333334
Macro F1: 0.2938830064261873
Weighted F1: 0.3440214692512549
              precision    recall  f1-score   support

        down     0.4106    0.0693    0.1186     37912
        flat     0.3796    0.1039    0.1631     28529
          up     0.4519    0.8926    0.6000     53559

    accuracy                         0.4450    120000
   macro avg     0.4140    0.3552    0.2939   

## Save results

In [8]:
metrics_dir = PROJECT_ROOT / "outputs" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []
for r in results:
    summary_rows.append({
        "model": r["model"],
        "accuracy": r["accuracy"],
        "macro_f1": r["macro_f1"],
        "weighted_f1": r["weighted_f1"],
        "train_seconds": r["train_seconds"],
        "train_samples": len(X_train),
        "val_samples": len(X_val),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False)
summary_path = metrics_dir / "baseline_summary.csv"
summary_df.to_csv(summary_path, index=False)

for r in results:
    name = r["model"].lower().replace(" ", "_")
    pd.DataFrame(
        r["confusion_matrix"],
        index=["true_down", "true_flat", "true_up"],
        columns=["pred_down", "pred_flat", "pred_up"],
    ).to_csv(metrics_dir / f"{name}_confusion_matrix.csv")

print("Saved:", rel_path(summary_path))
summary_df

Saved: outputs/metrics/baseline_summary.csv


,model,accuracy,macro_f1,weighted_f1,train_seconds,train_samples,val_samples
1,Logistic Regression,0.434333,0.322640,0.367895,1183.960537,250000,120000
2,Random Forest,0.444958,0.293883,0.344021,108.709810,250000,120000
0,Majority Class,0.446325,0.205728,0.275465,0.020739,250000,120000
